# Project for the Programming for Data Analytics Module

### Author: Kyra Menai Hamilton

Project specifications:
https://vlegalwaymayo.atu.ie/pluginfile.php/1804303/mod_resource/content/2/Project%20Description.pdf

## Packages

In [ ]:
# import the modules needed for analysis

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import requests
import io
import statsmodels.api as sm
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from scipy import stats
import os

## Ensuring File readability

I wanted to ensure that all the plots were legible and easy to read so I decided to standardise the font sizes across all text, axis labels, legends, and data labels within the figure.

In [ ]:
# Set global font sizes for better readability
plt.rcParams.update({
    'font.size': 14,
    'axes.labelsize': 16,
    'axes.titlesize': 18,
    'legend.fontsize': 12,
    'xtick.labelsize': 12,
    'ytick.labelsize': 12
})

Standardising of plot related text was conducted using the following sources as aid:
- [matplotlib - Text properties and layout](https://matplotlib.org/stable/users/explain/text/text_props.html)
- [Geeksforgeeks - Change Font Size in Matplotlib](https://www.geeksforgeeks.org/python/change-font-size-in-matplotlib/)
- [stackoverflow - How to change the font size on a matplotlib plot](https://stackoverflow.com/questions/3899980/how-to-change-the-font-size-on-a-matplotlib-plot)
- [Youtube - Properly Change Graph Size and Labels in Python and Matplotlib](https://www.youtube.com/watch?v=Fe3TBIAb6xE)
- [Youtube - Changing Font Properties in Matplotlib (Font Size & Family)](https://www.youtube.com/watch?v=EnQVUJsV7Lo)

## Importing Data

Before starting any data analysis, the data needed to be [saved as a csv](https://realpython.com/python-csv/) for easier data cleaning and analysis.
[Saving the data locally as a csv](https://docs.python.org/3/library/csv.html) allowed for more convenient data access, and the [pandas module was used to read the csv file](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.read_csv.html).

In [ ]:
# Load the data from the CSO API and save the dataset as a CSV file.
url = "https://ws.cso.ie/public/api.jsonrpc?data=%7B%22jsonrpc%22:%222.0%22,%22method%22:%22PxStat.Data.Cube_API.ReadDataset%22,%22params%22:%7B%22class%22:%22query%22,%22id%22:%5B%5D,%22dimension%22:%7B%7D,%22extension%22:%7B%22pivot%22:null,%22codes%22:false,%22language%22:%7B%22code%22:%22en%22%7D,%22format%22:%7B%22type%22:%22CSV%22,%22version%22:%221.0%22%7D,%22matrix%22:%22AFA01%22%7D,%22version%22:%222.0%22%7D%7D"
response = requests.get(url)
json_data = response.json()
csv_string = json_data['result']
data = pd.read_csv(io.StringIO(csv_string))
try:
    script_dir = os.path.dirname(os.path.abspath(__file__))
except NameError:
    script_dir = os.getcwd()  # For Jupyter notebooks
csv_path = os.path.join(script_dir, 'pfda_data.csv')
data.to_csv(csv_path, index=False)
# note if saved successfully, a file named 'pfda_data.csv' will appear in the working directory

The data was afforestation data, but for the purposes of this project, it was saved as [pfda_data.csv](https://github.com/KaiiMenai/programming-for-data-analytics/blob/main/pfda-project/pfda_data.csv).

## Ensuring Folders are ready for outputs

I created folders for the outputs of the analysis; [basic_statistical_analysis](https://github.com/KaiiMenai/programming-for-data-analytics/tree/main/pfda-project/basic_statistical_analysis) for the basic statistical analysis data csv files and markdown output, [individual_plots](https://github.com/KaiiMenai/programming-for-data-analytics/tree/main/pfda-project/outputs/individual_plots) for the outputs of the top 5 counties for afforestation looking at forest owner trends, species trends, and value over the years (hectares), and [outputs](https://github.com/KaiiMenai/programming-for-data-analytics/tree/main/pfda-project/outputs) for all analysis outputs for the remaining analysis.

In [ ]:
# Create a subfolder for outputs
outputs_dir = os.path.join(script_dir, 'outputs')
os.makedirs(outputs_dir, exist_ok=True)

# Create a subfolder for basic statistical analysis
basic_analysis_dir = os.path.join(script_dir, 'basic_statistical_analysis')
os.makedirs(basic_analysis_dir, exist_ok=True)

There are a few ways of creating sub-folders in github, one is [directly using the buttons and options](https://docs.github.com/en/repositories/working-with-files/managing-files/creating-new-files), another involves [writing directly into the python script](https://stackoverflow.com/questions/35950556/python-create-folder-with-multiple-subfolders) (this is what I did).

## Reading, Cleaning, Drop Rows - Checking the Data

I checked the data by getting the first few rows printed out. 
Prior to analysing the data, it needed to be [cleaned](https://realpython.com/python-data-cleaning-numpy-pandas/) and checked for any missing values, missing values would be removed.



In [ ]:
# Read the dataset
data = pd.read_csv(csv_path)
# Display the first few rows of the dataset
print(data.head())

# Now to clean the dataset
# Check for missing values
print(data.isnull().sum())

# Drop rows with missing values
data = data.dropna()
# Verify that there are no more missing values
print(data.isnull().sum())
# Display data types of each column
print(data.dtypes)

# For now I think the data types are fine, but if needed we can convert them later.

## Basic Summary Statistics

[Summary statistics](https://realpython.com/python-statistics/) and basic [descriptive statistics](https://snakebear.science/08-Statistics/08_1_Basic_Statistics_Descriptives.html) was conducted on the on the dataset.

In [ ]:
# Basic Data Exploration
# Summary Statistics for value vs county vs year
summary_stats = data.groupby(['County', 'Year'])['VALUE'].describe()
print(summary_stats)
# Save summary statistics to a CSV file
summary_stats_path = os.path.join(basic_analysis_dir, 'summary_statistics.csv')
summary_stats.to_csv(summary_stats_path)

Observations

## Basic Data Analysis

For the basic analysis of the data, I split my approach to look at the data in more depth for different variables.

For each of the part of the basic data analysis, I focused on using a variety of techniques in order to see any valuable insights into the data.

1. Average VALUE (hectares) per county
    - This analysis looked at the central tendency of afforestation values across years for each county ([groupby method](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.groupby.html)), this helps with identifying which counties have consistently higher or lower afforestation activity. This allows for comparing the overall performance without it being skewed by outliers within the dataset.
    - I used `counties_data.groupby('County')['VALUE'].mean()` to group by county and compute the mean value.
2. Total VALUE (hectares) per county
    - This [summed up all the afforestation values over time for each of the counties](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.groupby.html), this provided a cumulative measure of the total activity. This analysis highlights the counties with the greatest overall contribution to afforestation, this was useful to aid in resource allocation or policy decisions.
    - I use `counties_data.groupby('County')['VALUE'].sum()` to give a sum total for each of the counties.
3. Coefficient of Variation per county
    - The [coefficient of variation calculates the relative variability of the data](https://en.wikipedia.org/wiki/Coefficient_of_variation) (calculated by: standard deviation divided by the mean), this is useful as it shows how consistent afforestation was year-to-year. The higher the value, the more inconsistent the values in the dataset - this could suggest external factors may have impacted afforestation (such as environmental factors, or policy changes).
    - I used `(counties_data.groupby('County')['VALUE'].std() / counties_data.groupby('County')['VALUE'].mean())` to [calculate the coefficient of variation](https://numpy.org/doc/stable/reference/routines.statistics.html).
4. Maximum VALUE (hectares) per county
    - This will identify the max afforestation in any one year per county, this will aid in revealing outliers or years with unusually high afforestation. From a policy perspective this would aid in spotting years with exceptional afforestation and how to go about looking at strategies for maintaining high levels of afforestation.
    - To do this, I used `counties_data.groupby('County')['VALUE'].max()` [to find the highest value per group](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.groupby.html).
5. Number of years with data per county
    - This counts the number of years with data for each county, this helps with seeing how complete the dataset is, and in turn, how reliable the dataset was. Counties with fewer years of data, may be due to incomplete because of reporting, and this may affect the accuracy of the analysis in this project.
    - To [count the number of unique years of data](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.groupby.html), I used `counties_data.groupby('County')['Year'].nunique()`
6. Growth Rate per county
    - This looked at the [simplified percentage change](https://www.investopedia.com/terms/g/growthrates.asp) from the first to last year of data, indicating any trends in afforestation over time. For example, positive rates show growth, which in turn is useful for forecasting and evaluating progress towards a specific goal.
    - For this I needed to define a function that would calculate `(last - first) / first` for each county's sorted data, then [apply this to the dataset](https://www.investopedia.com/terms/g/growthrates.asp) using `groupby().apply()`.

In [ ]:
# Look at interesting statistics findings for the counties and write output to a markdown (.md) file.
with open(os.path.join(basic_analysis_dir, "basic_analysis.md"), "w") as file:
    print("# Basic Statistical Analysis on Ireland Afforestation Data", file=file)
    print("\n***Author: Kyra Menai Hamilton***", file=file)
    print("\n## Summary", file=file)
    summary_text = (
        print("\n## Interesting Statistics Findings for Counties \n", file=file)
    )
    print("\n1. Average VALUE per county (excluding Ireland for county-specific analysis)", file=file)
    counties_data = data[data['County'] != 'Ireland']
    avg_value_per_county = counties_data.groupby('County')['VALUE'].mean().sort_values(ascending=False)
    print(avg_value_per_county.head(10).to_markdown(), file=file)
    avg_value_path = os.path.join(basic_analysis_dir, 'avg_value_per_county.csv')
    avg_value_per_county.to_csv(avg_value_path)

    print("\n### 2. Total VALUE per County", file=file)
    print("Total afforestation value summed across all years.", file=file)
    total_value_per_county = counties_data.groupby('County')['VALUE'].sum().sort_values(ascending=False)
    print(total_value_per_county.head(10).to_markdown(), file=file)
    total_value_path = os.path.join(basic_analysis_dir, 'total_value_per_county.csv')
    total_value_per_county.to_csv(total_value_path)

    print("\n### 3. Coefficient of Variation per County", file=file)
    print("Measures relative variability (std/mean); higher values indicate more year-to-year inconsistency.", file=file)
    cv_per_county = (counties_data.groupby('County')['VALUE'].std() / counties_data.groupby('County')['VALUE'].mean()).sort_values(ascending=False)
    print(cv_per_county.head(10).to_markdown(), file=file)
    cv_path = os.path.join(basic_analysis_dir, 'cv_per_county.csv')
    cv_per_county.to_csv(cv_path)

    print("\n### 4. Maximum VALUE per County", file=file)
    print("Peak afforestation value in any single year.", file=file)
    max_value_per_county = counties_data.groupby('County')['VALUE'].max().sort_values(ascending=False)
    print(max_value_per_county.head(10).to_markdown(), file=file)
    max_value_path = os.path.join(basic_analysis_dir, 'max_value_per_county.csv')
    max_value_per_county.to_csv(max_value_path)

    print("\n### 5. Number of Years with Data per County", file=file)
    print("Indicates data completeness for each county.", file=file)
    years_per_county = counties_data.groupby('County')['Year'].nunique().sort_values(ascending=False)
    print(years_per_county.head(10).to_markdown(), file=file)
    years_path = os.path.join(basic_analysis_dir, 'years_per_county.csv')
    years_per_county.to_csv(years_path)

    print("\n### 6. Growth Rate per County", file=file)
    print("Simplified growth rate: (last year's VALUE - first year's VALUE) / first year's VALUE.", file=file)
    def growth_rate(group):
        if len(group) < 2:
            return np.nan
        first = group['VALUE'].iloc[0]
        last = group['VALUE'].iloc[-1]
        return (last - first) / first if first != 0 else np.nan
    growth_per_county = counties_data.sort_values('Year').groupby('County').apply(growth_rate).sort_values(ascending=False)
    print(growth_per_county.head(10).to_markdown(), file=file)
    growth_path = os.path.join(basic_analysis_dir, 'growth_rate_per_county.csv')
    growth_per_county.to_csv(growth_path)

    print("\n## Data Sources and Outputs", file=file)
    print("- Raw data: pfda_data.csv", file=file)
    print("- Summary statistics: summary_statistics.csv", file=file)
    print("- All statistical CSVs are in the basic_statistical_analysis/ folder.", file=file)
    print("- Plots are saved in outputs/ as PNG files.", file=file)

print("Interesting findings written to basic_statistical_analysis/basic_analysis.md") # Ref of how to do all this is from my pands-project: https://github.com/KaiiMenai/pands-project/blob/main/analysis.py


Observations

## Analysis

### Plot for visualising values and their frequency

Looking at the frequency of values (hectares) recorded for the years.

In [ ]:
# Visualise distributions of key variables
plt.figure(figsize=(10, 6))
sns.histplot(data['VALUE'], bins=30, kde=True)
plt.title('Distribution of Values')
plt.xlabel('Value (hectares)')
plt.ylabel('Frequency')
plt.show()
# Save the plot
hist_path = os.path.join(outputs_dir, 'value_distribution.png')
plt.savefig(hist_path)

From the plot, it can be seen that the majority of the values for the number of hectares planted on any given county on any given year were under 2000 hectares, with most being under 1000 hectares.

### All Ireland Afforestation Value over the Years

To see the value (hectares) planted over the years for Ireland as a whole, I filtered the data to specifically show only the afforestation labelled as Ireland. 

In [ ]:
# Visualise the Value over the years for Ireland as a whole
ireland_data = data[data['County'] == 'Ireland']
plt.figure(figsize=(12, 6))
sns.lineplot(x='Year', y='VALUE', data=ireland_data, marker='o')
plt.title('Afforestation value over Years for Ireland')
plt.xlabel('Year')
plt.ylabel('Value (hectares)')
plt.grid()
plt.show()
# Save the plot
lineplot_path = os.path.join(outputs_dir, 'value_over_years_ireland.png')
plt.savefig(lineplot_path)

The defined line is the mean value for the values for Ireland for each year. The highlighted area around the mean line is the 95 % confidence interval (ci) around the mean.

It can clearly be seen that for Ireland as a whole, there has been a continuing decrease in value (hectares) for afforestation since 2016.

### Species Afforestation for Ireland as a whole

The species trends for afforestation for Ireland as a whole was also analysed. The Total Afforestation value was included in order to make the species breakdown comparison easier.

In [ ]:
# Analyse data over the years for Ireland as a whole. Look at species trends.
species_trends = ireland_data.groupby(['Year', 'Species'])['VALUE'].sum().reset_index()
plt.figure(figsize=(35, 15))
sns.lineplot(x='Year', y='VALUE', hue='Species', data=species_trends, marker='o')
plt.title('Species Trends over Years for Ireland')
plt.xlabel('Year')
plt.ylabel('Value (hectares)')
plt.legend(title='Species', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid()
plt.tight_layout()
plt.show()
# Save the plot
species_trends_path = os.path.join(outputs_dir, 'species_trends_ireland.png')
plt.savefig(species_trends_path)    

From the plot, it can be seen, that over the years, afforestation of the Broadleaf species has been lower than for the Conifer species. In 2023, the value (hectares) of afforestation for Broadleaf was greater than that of the Conifer species. The value (hectares) of Broadleaf afforestation has decreased steadily since 2010. In contrast, Conifer species afforestation remained steady until 2016, from which it decreased rapidly over the first few years, and had a slower reduction from 2018 onwards.

### Correlation Matrix

In [ ]:
# I wanted to look to see if there were any correlations between the species planted and the Forest owner.
correlation_data = data[['Species', 'Forest Owner', 'VALUE']]#
correlation_data = correlation_data.dropna()
correlation_data['Species_Code'] = correlation_data['Species'].astype('category').cat.codes
correlation_data['Forest_Owner_Code'] = correlation_data['Forest Owner'].astype('category').cat.codes
correlation_matrix = correlation_data[['Species_Code', 'Forest_Owner_Code', 'VALUE']].corr()
print(correlation_matrix)
# Visualize the correlation matrix
plt.figure(figsize=(8, 6))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt=".2f")
plt.title('Correlation Matrix')
plt.show()
# Save the plot
correlation_matrix_path = os.path.join(outputs_dir, 'correlation_matrix.png')
plt.savefig(correlation_matrix_path)    
# Save the correlation matrix to a CSV file
correlation_matrix_csv_path = os.path.join(outputs_dir, 'correlation_matrix.csv')
correlation_matrix.to_csv(correlation_matrix_csv_path)

The Correlation Matrix shows no correlation between the species planted and the forest owner.

### Forest Owner over the Years

Looking at the forest owner over the years in Ireland for total afforestation.

In [ ]:
# Now for analysis for the forest owner types over the years for Ireland as a whole
forest_owner_trends = ireland_data.groupby(['Year', 'Forest Owner'])['VALUE'].sum().reset_index()
plt.figure(figsize=(35, 15))
sns.lineplot(x='Year', y='VALUE', hue='Forest Owner', data=forest_owner_trends, marker='o')
plt.title('Forest Owner Trends over Years for Ireland')
plt.xlabel('Year')
plt.ylabel('Value (hectares)')
plt.legend(title='Forest Owner', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid()
plt.tight_layout()
plt.show()
# Save the plot
forest_owner_trends_path = os.path.join(outputs_dir, 'forest_owner_trends_ireland.png')
plt.savefig(forest_owner_trends_path)

Forest Owner in Ireland was classed as one of 3 groups: public sector (red), farmer (blue), and non-farmer (orange). 
Up until 2018, Farmer remained the largest value Forest Owner in Ireland. From 2018 onwards, Non-Farmer owners became the largest value Forest Owner in Ireland. Public Sector Forest Owners have remained at a consistently low value.

The data suggests that up until 2018, Farmer Forest Owner(s) were responsible for the majority of the afforestation value (hectares) in Ireland.

## Specific County Analysis

Breakdown the counties and remove Ireland as an option.

In [ ]:
# Now I want to analyse the data for specific counties. I want to look at the top 5 counties with the highest total value over the years.
county_totals = data[data['County'] != 'Ireland'].groupby('County')['VALUE'].sum().reset_index()
top_5_counties = county_totals.nlargest(5, 'VALUE')['County'].tolist()
print("Top 5 Counties with highest total value over the years:", top_5_counties)

For easier visual interpretation I decided to overlay the plots for the top 5 counties.

In [ ]:
# Overlay plots for value over years for top 5 counties
top_5_data = data[data['County'].isin(top_5_counties)]
plt.figure(figsize=(12, 6))
sns.lineplot(x='Year', y='VALUE', hue='County', data=top_5_data, marker='o')
plt.title('Afforestation Value over Years for Top 5 Counties')
plt.xlabel('Year')
plt.ylabel('Value (hectares)')
plt.grid()
plt.show()
# Save the plot
overlay_plot_path = os.path.join(outputs_dir, 'value_over_years_top5_counties.png')
plt.savefig(overlay_plot_path)

This plot shows the mean value (hectares) and 95 % confidence interval for afforestation for the top 5 counties (Clare, Cork, Kerry, Meath, and Roscommon) from 2007 to 2023.

It can be seen that Co. Cork has continuously had the highest value for afforestation.
Co. Roscommon was the lowest (of the top 5) between 2007 and 2012, and again in 2015. Co. Mayo had the lowest value in 2013, 2018. 2019, and 2021. The lowest value in 2016 and 2017 was Co. Kerry, where Co. Clare had the lowest value in 2020 only. *lowest value here is out of the top 5 counties.

## County comparison

I wanted to do multiple types of plots with the data to aid in visual interpretation.
For these plots each county was created and saved individually. The first is looking directly at the afforestation value for each year, the second is looking at the Species trends in the counties, whether Conifer or Broadleaf was planted. The third is looking at the Forest Owner trends in the county, Forest Owner was defined as Farmer, Non-Farmer, or Public Sector.

In [ ]:
for county in top_5_counties:
    county_data = data[data['County'] == county]
    
    # Create a combined subplot for the county: 1 row, 3 columns
    fig, axes = plt.subplots(1, 3, figsize=(25, 6))
    fig.suptitle(f'Afforestation Trends for {county}', fontsize=16)
    
    # Left subplot: Value over Years
    sns.lineplot(x='Year', y='VALUE', data=county_data, marker='o', ax=axes[0])
    axes[0].set_title('Value over Years')
    axes[0].set_xlabel('Year')
    axes[0].set_ylabel('Value (hectares)')
    axes[0].grid()
    
    # Middle subplot: Species Trends
    species_trends_county = county_data.groupby(['Year', 'Species'])['VALUE'].sum().reset_index()
    sns.lineplot(x='Year', y='VALUE', hue='Species', data=species_trends_county, marker='o', ax=axes[1])
    axes[1].set_title('Species Trends')
    axes[1].set_xlabel('Year')
    axes[1].set_ylabel('Value (hectares)')
    axes[1].legend(title='Species', bbox_to_anchor=(1.05, 1), loc='upper left')
    axes[1].grid()
    
    # Right subplot: Forest Owner Trends
    forest_owner_trends_county = county_data.groupby(['Year', 'Forest Owner'])['VALUE'].sum().reset_index()
    sns.lineplot(x='Year', y='VALUE', hue='Forest Owner', data=forest_owner_trends_county, marker='o', ax=axes[2])
    axes[2].set_title('Forest Owner Trends')
    axes[2].set_xlabel('Year')
    axes[2].set_ylabel('Value (hectares)')
    axes[2].legend(title='Forest Owner', bbox_to_anchor=(1.05, 1), loc='upper left')
    axes[2].grid()
    
    plt.tight_layout()
    plt.show()
    # Save the combined plot for the county
    combined_county_path = os.path.join(outputs_dir, f'combined_{county.replace(" ", "_").lower()}.png')
    plt.savefig(combined_county_path)
    plt.close() # Got errors about memory use, so best to close the plots after saving.

As multiple plots can be difficult to refer between for visualising trends, I decided to create one plot with multiple sub-plots to give a better visualisation for the trends seen in the top 5 counties (Counties; Clare, Cork, Kerry, Mayo, and Roscommon). 

I used the basis from my [pands project](https://github.com/KaiiMenai/pands-project/blob/main/analysis.py) where I made multiple subplots as a basis for this. Further reference aid was made from [matplotlib](https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.subplots.html) and [seaborn](https://seaborn.pydata.org/tutorial/aesthetics.html#seaborn-figure-styles) to aid in readability.

In [ ]:
# Combined subplots for top 5 counties: value over years, species trends, and forest owner trends
fig, axes = plt.subplots(5, 3, figsize=(25, 25))
fig.suptitle('Afforestation Trends for Top 5 Counties', fontsize=20)

for i, county in enumerate(top_5_counties):
    county_data = data[data['County'] == county]
    
    # Left subplot: Value over Years
    ax1 = axes[i, 0]
    sns.lineplot(x='Year', y='VALUE', data=county_data, marker='o', ax=ax1)
    ax1.set_title(f'Value over Years for {county}')
    ax1.set_xlabel('Year')
    ax1.set_ylabel('Value (hectares)')
    ax1.grid()
    
    # Middle subplot: Species Trends
    ax2 = axes[i, 1]
    species_trends_county = county_data.groupby(['Year', 'Species'])['VALUE'].sum().reset_index()
    sns.lineplot(x='Year', y='VALUE', hue='Species', data=species_trends_county, marker='o', ax=ax2)
    ax2.set_title(f'Species Trends over Years for {county}')
    ax2.set_xlabel('Year')
    ax2.set_ylabel('Value (hectares)')
    ax2.legend(title='Species', bbox_to_anchor=(1.05, 1), loc='upper left')
    ax2.grid()
    
    # Right subplot: Forest Owner Trends
    ax3 = axes[i, 2]
    forest_owner_trends_county = county_data.groupby(['Year', 'Forest Owner'])['VALUE'].sum().reset_index()
    sns.lineplot(x='Year', y='VALUE', hue='Forest Owner', data=forest_owner_trends_county, marker='o', ax=ax3)
    ax3.set_title(f'Forest Owner Trends over Years for {county}')
    ax3.set_xlabel('Year')
    ax3.set_ylabel('Value (hectares)')
    ax3.legend(title='Forest Owner', bbox_to_anchor=(1.05, 1), loc='upper left')
    ax3.grid()

plt.tight_layout()
plt.show()
# Save the combined plot
combined_plot_path = os.path.join(outputs_dir, 'combined_trends_top5_counties.png')
plt.savefig(combined_plot_path)
plt.close('all')  # I kept getting alerts for memory use so I added this as it tells py to close all figures to free memory


From the top 5, it can be seen that there were some counties where Public Sector afforestation value was not recored/included/missing; out of the top 5 counties, Co. Cork, Co. Clare, and Co. Kerry had no data on Public Sector Forest Owner(s).

#### How I Created the Combined Subplots:

1. **Set up the figure and axes**: Used `plt.subplots(5, 3, figsize=(25, 25))` to create a 5 x 3 grid (5 counties, 3 plot types each).
2. **Added a main title**: `fig.suptitle('Afforestation Trends for Top 5 Counties', fontsize=20)` for the overall figure.
3. **Looped through counties**: For each of the top 5 counties, filtered the data and created three subplots per row.
4. **Left subplot (Value over Years)**: Simple line plot of VALUE vs Year for the county.
5. **Middle subplot (Species Trends)**: Grouped data by Year and Species, then plotted VALUE vs Year with Species as hue.
6. **Right subplot (Forest Owner Trends)**: Grouped data by Year and Forest Owner, then plotted VALUE vs Year with Forest Owner as hue.
7. **Customized each subplot**: Added titles, labels, grids, and legends (positioned outside to avoid overlap).
8. **Adjusted layout**: `plt.tight_layout()` to prevent overlapping, then saved with `plt.savefig()`.
9. **Closed figures**: Added `plt.close('all')` to free memory after saving.

This approach makes it easy to compare trends across counties and variables in one view.

### Plot for all counties - exclude Ireland

I wanted to see a visual of all the mean afforestation values (hectares) without the confidence intervals, to get a clear picture of afforestation trends for all the counties.

In [ ]:
# Plot mean VALUE over years for all counties (excluding Ireland)
counties_data = data[data['County'] != 'Ireland']
mean_value_per_year_county = counties_data.groupby(['Year', 'County'])['VALUE'].mean().reset_index()
plt.figure(figsize=(30, 15))
sns.lineplot(x='Year', y='VALUE', hue='County', data=mean_value_per_year_county, marker='o')
plt.title('Mean Afforestation Value over Years for All Counties')
plt.xlabel('Year')
plt.ylabel('Mean Value (hectares)')
plt.legend(title='County', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid()
plt.tight_layout()
plt.show()
# Save the plot
all_counties_plot_path = os.path.join(outputs_dir, 'mean_value_over_years_all_counties.png')
plt.savefig(all_counties_plot_path)

Overall, it can be seen that Co. Cork has a consistently higher afforestation value (hectares) than all other counties (apart from in 2017 and 2018).

## Further Analysis

### Afforestation over the years in Ireland

This plot is repeated from the initial basic analysis, I decided to show it here to refresh for the next analysis but here I removed the shaded area showing the 95 % confidence interval.

In [ ]:
# Visualise the Value over the years for Ireland as a whole
ireland_data = data[data['County'] == 'Ireland']
plt.figure(figsize=(12, 6))
sns.lineplot(x='Year', y='VALUE', data=ireland_data, marker='o', ci=None)
plt.title('Afforestation value over Years for Ireland')
plt.xlabel('Year')
plt.ylabel('Value (hectares)')
plt.grid()
plt.show()
# Save the plot
lineplot_path = os.path.join(outputs_dir, 'value_over_years_ireland.png')
plt.savefig(lineplot_path)

From the basic plot for the mean afforestation value for Ireland, it can be seen that there is a difference between the value (hectares) recorded for the first year (2007) and the most recent year (2023) recorded. I wanted to analyse this further to see if the difference was significant.
To see if there was a significant I decided to run a [simple T-test](https://www.datacamp.com/tutorial/an-introduction-to-python-t-tests). 

In [ ]:
# I want to look to see if there has been a significant difference in afforestation between the first year and most recent year.
first_year = data['Year'].min()
most_recent_year = data['Year'].max()   
first_year_data = data[data['Year'] == first_year]['VALUE']
most_recent_year_data = data[data['Year'] == most_recent_year]['VALUE']
t_stat, p_value = stats.ttest_ind(first_year_data, most_recent_year_data)
print(f"T-statistic: {t_stat}, P-value: {p_value}")
if p_value < 0.05:
    print("There is a significant difference in afforestation between the first year and the most recent year.")
else:
    print("There is no significant difference in afforestation between the first year and the most recent year.")

In a [t-test](https://www.geeksforgeeks.org/data-science/t-test/) it is important to look at the p value output to see if there is a significant difference. A significant difference is any p-value < 0.05 as this signals that there is less than 5 % chance of error. I used the [`ttest_ind`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.ttest_ind.html) in the scipy module. 

From the results of the t-test, the p-value was 0.000000? meaning that there was a significant difference between the first and last value recorded.

I wanted to make a boxplot to visualise the differences between the first and last recorded afforestation values for Ireland. I decided to use a [boxplot](https://www.atlassian.com/data/charts/box-plot-complete-guide) because it would allow for seeing the afforestation values for both the first (2007) and last year (2023), highlighting the interquartile range, the median, the upper and lower quartiles, as well as any outliers within the data. 

In [ ]:
# Since there is a difference in the afforestation between the first and most recent year, I want to visualise this using boxplots.
plt.figure(figsize=(10, 6))
sns.boxplot(x='Year', y='VALUE', data=data[data['Year'].isin([first_year, most_recent_year])])
plt.title('Afforestation Values: First Year vs Most Recent Year')
plt.xlabel('Year')
plt.ylabel('Value')
plt.show()
# Save the plot
boxplot_path = os.path.join(outputs_dir, 'afforestation_boxplot.png')
plt.savefig(boxplot_path)

After seeing that there were differences in the afforestation between the first year recorded and the most recent recorded year, I wanted to see if there were any trends in the species being planted and the forest owner.

In [ ]:
# Visualise forest owner trends vs species trends for Ireland.
ireland_data = data[data['County'] == 'Ireland']
forest_owner_trends_ireland = ireland_data.groupby(['Year', 'Forest Owner'])['VALUE'].sum().reset_index()
species_trends_ireland = ireland_data.groupby(['Year', 'Species'])['VALUE'].sum().reset_index()

First to look at Forest Owner (included here for clarity, is already included at the beginning).

In [ ]:
plt.figure(figsize=(20, 12))
sns.lineplot(x='Year', y='VALUE', hue='Forest Owner', data=forest_owner_trends_ireland, marker='o')
plt.title('Forest Owner Trends over Years for Ireland')
plt.xlabel('Year')
plt.ylabel('Value (hectares)')
plt.legend(title='Forest Owner', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid()
plt.tight_layout()
plt.show()
# Save the plot
forest_owner_trends_ireland_path = os.path.join(outputs_dir, 'forest_owner_trends_ireland.png')
plt.savefig(forest_owner_trends_ireland_path)

Then Species (again this is done at the start as well, but included here for clarity).

In [ ]:
plt.figure(figsize=(20, 12))
sns.lineplot(x='Year', y='VALUE', hue='Species', data=species_trends_ireland, marker='o')
plt.title('Species Trends over Years for Ireland')
plt.xlabel('Year')
plt.ylabel('Value (hectares)')
plt.legend(title='Species', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid()
plt.tight_layout()
plt.show()
# Save the plot
species_trends_ireland_path = os.path.join(outputs_dir, 'species_trends_ireland.png')
plt.savefig(species_trends_ireland_path)

I wanted to run it directly to see if there was a correlation between forest owner and the species being planted. I decided to use a [linear regression model](https://www.datacamp.com/tutorial/linear-regression-in-python). For making the code for the linear regression I modified the code found on [Medium](https://kirannagarkoti.medium.com/linear-regression-explained-assumptions-interpretation-python-implementation-ee2db5c885bb) and also used [Geeksforgeeks](https://www.geeksforgeeks.org/machine-learning/interpreting-the-results-of-linear-regression-using-ols-summary/), [DataCamp](https://www.datacamp.com/tutorial/linear-regression-in-python), and [Real Python](https://realpython.com/linear-regression-in-python/) to aid in interpretation and running the code.

In [ ]:
# Now that I've done some analysis, I want to further explore using a linear regression model and to see if I can predict afforestation values based on year, species, forest owner, and county.
# Now to prepare the data for modeling.
model_data = data.copy()
# Encode categorical variables - https://scikit-learn.org/stable/modules/preprocessing.html#encoding-categorical-features
model_data = pd.get_dummies(model_data, columns=['Species', 'Forest Owner', 'County'])  # Include all dummies
# Define features and target variable
X = model_data.drop(columns=['VALUE', 'Year', 'Statistic Label', 'UNIT'])  # Exclude 'Year', and non-numeric columns
y = model_data['VALUE']
# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42) # test size of 20%
# Create and train the linear regression model
model = LinearRegression()
model.fit(X_train, y_train)
# Make predictions
y_pred = model.predict(X_test)
# Evaluate the model
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
print(f"Mean Squared Error: {mse}")
print(f"R-squared: {r2}")

As R-squared is 0.439, there is no correlation between the type of tree species being planted and forest owner.

Can the model predict 2030 afforestation hectarage?

To ensure that the data has unique series in species and forest owner I modified [`unique_series = sorted(data['SERIES'].unique())`](https://pandas.pydata.org/docs/reference/api/pandas.Series.unique.html).

In [ ]:
# Now to see if the model can predict afforestation values for a specific year, species, forest owner, and county. Given that Ireland is aiming for 8,000 hectares of new afforestation annually by 2030, I will use this as a test case.

# Get unique species and forest owners from the data
unique_species = sorted(data['Species'].unique())
unique_forest_owners = sorted(data['Forest Owner'].unique())

print("\nUnique Species in dataset:")
print(unique_species)
print("\nUnique Forest Owners in dataset:")
print(unique_forest_owners)

Creating the test case.
I used the for and [if condition](https://stackoverflow.com/questions/60799900/for-loop-with-dictionary-and-if-condition) to make a loop for the species, forest owner, and county (county is just Ireland in this case).


In [ ]:
# Build test case for total afforestation in Ireland for 2030
test_case = {}

# Set species to 'Total Afforestation'
for species in unique_species:
    col_name = f'Species_{species}'
    test_case[col_name] = 1 if species == 'Total Afforestation' else 0

# Set forest owner to 'Total Afforestation'
for owner in unique_forest_owners:
    col_name = f'Forest Owner_{owner}'
    test_case[col_name] = 1 if owner == 'Total Afforestation' else 0

# Set county to 'Ireland'
unique_counties = sorted(data[data['County'] != 'Ireland']['County'].unique())
for county in unique_counties:
    col_name = f'County_{county}'
    test_case[col_name] = 0
test_case['County_Ireland'] = 1  # Ireland is not in unique_counties, so add manually

# Ensure all model features are in the test case with default values of 0
for feature in X.columns:
    if feature not in test_case:
        test_case[feature] = 0

test_case_df = pd.DataFrame([test_case])
predicted_value = model.predict(test_case_df)
print(f"\nPredicted total afforestation value for Ireland in 2030: {predicted_value[0]:.2f} hectares")
if predicted_value[0] >= 8000:
    print("The predicted value meets or exceeds the target of 8000 hectares per year.")
else:
    print("The predicted value is below the target of 8000 hectares per year.")

In the dataset, 'Ireland' is not just the sum of individual counties, it's a separate category provided by the CSO, where its value represents the **national total afforestation**. This is why predicting for `County_Ireland = 1` is done. The model is trained on historical national totals and directly predicts the future total for Ireland as a whole.

To verify that the values were correct: Ireland's values in the data match the sums of all counties for the same year, species, and forest owner combinations. Using the Ireland dummy ensures that the model is predicting the overall total, not summing individual county predictions, which could potentially miss interactions in the model.

From the test model, it can be seen that Ireland would not reach the required afforestation value of 8000 hectares per annum.

### But where am I getting these values from?

Ireland has a goal to increase National Forest Cover from 11.6 % (in 2024) to 18 % by 2050, to offset carbon emissions. This will require a National afforestation value of 8000 hectares per annum.

References for afforestation requirements:

- [Citizens Information - Afforestation Scheme](https://www.citizensinformation.ie/en/environment/land/afforestation-scheme/) 
- [Gov.ie - Forestry](https://www.gov.ie/en/department-of-agriculture-food-and-the-marine/press-releases/shared-national-vision-for-forestry-2050-published/)
- [Gov.ie - Ireland's Forestry Strategy](https://www.gov.ie/en/department-of-agriculture-food-and-the-marine/publications/irelands-forest-strategy-2023-2030/)
- [Forestry Services - Planting Rates](https://forestryservices.ie/planting-rates-must-exceed-8000ha-per-year/)
- [Independent.ie - Forestry](https://www.independent.ie/farming/forestry-enviro/forestry-planting-rates-must-exceed-8000ha-per-year-to-meet-climate-targets-epa/a1011415871.html)
- [Coillte](https://www.coillte.ie/coillte-launches-new-forestry-strategic-vision-to-optimise-its-contribution-to-irelands-climate-targets/#:~:text=28m%20tonnes%20of%20CO2,which%20will%20be%20native%20woodlands) (Non-Farmer private owner - but also kind of a public body) aims to plant 100,000 hectares of forest by 2050.

# END